# Data Build: Modelling Dataset

This notebook constructs the modelling dataset used throughout the project.

It combines:
- South West demand proxy data
- GB wind generation data

Data is aligned to settlement period level using a common sp_key.

Key steps:
- Standardise keys across datasets
- Aggregate wind to settlement period level
- Merge demand and wind
- Handle missing values via limited interpolation
- Validate settlement period completeness

Output:
data/processed/modelling_dataset.csv

## 1. Load Data

In [2]:
import pandas as pd

# Paths
DATA_RAW = "../data/raw/"
DATA_INTERIM = "../data/interim/"
DATA_PROCESSED = "../data/processed/"


wind = pd.read_parquet(DATA_RAW + "wind_history_all_vintages.parquet")
demand = pd.read_csv(DATA_RAW + "sw_gsp_demand_proxy.csv")

print("Wind:", wind.shape)
print("Demand:", demand.shape)

import os
os.makedirs(DATA_PROCESSED, exist_ok=True)

Wind: (656188, 5)
Demand: (17520, 4)


## 2. BUILD DEMAND SP_KEY (MASTER TIMELINE)

In [3]:
demand = demand.rename(columns={
    'settlementDate': 'settlement_date',
    'settlementPeriod': 'settlement_period'
})

demand['settlement_date'] = pd.to_datetime(demand['settlement_date']).dt.date
demand['settlement_period'] = demand['settlement_period'].astype(int)

demand['sp_key'] = (
    demand['settlement_date'].astype(str)
    + "_"
    + demand['settlement_period'].astype(str)
)

df = demand.copy()

## 3. PREP WIND DATA

In [4]:
wind['publish_time'] = pd.to_datetime(wind['publish_time'])
wind['start_time'] = pd.to_datetime(wind['start_time'])

# Lead time in hours
wind['lead_time_hours'] = (
    (wind['start_time'] - wind['publish_time'])
    .dt.total_seconds() / 3600
)

## 4. FILTER TO INTRADAY FORECASTS

In [5]:
wind_filtered = wind[
    (wind['lead_time_hours'] >= -2) &
    (wind['lead_time_hours'] <= 6)
].copy()

print("Filtered wind rows:", len(wind_filtered))

Filtered wind rows: 44018


## 5. COLLAPSE TO ONE ROW PER SP

In [6]:
# Clean types
wind_filtered['settlement_date'] = pd.to_datetime(
    wind_filtered['settlement_date']
).dt.date

wind_filtered['settlement_period'] = wind_filtered['settlement_period'].astype(int)

# Sort by recency
wind_filtered = wind_filtered.sort_values(
    ['settlement_date', 'settlement_period', 'publish_time']
)

# Collapse
wind_clean = (
    wind_filtered
    .groupby(['settlement_date', 'settlement_period'], as_index=False)
    .last()
)

In [7]:
wind_clean['sp_key'] = (
    wind_clean['settlement_date'].astype(str)
    + "_"
    + wind_clean['settlement_period'].astype(str)
)

## 6. MERGE WIND ONTO DEMAND

In [8]:
df = df.merge(
    wind_clean[['sp_key', 'generation']],
    on='sp_key',
    how='left'
)

df = df.rename(columns={'generation': 'wind_mw'})

Wind forecasts are sparse (~80% missing after filtering).
We interpolate to approximate continuous physical generation.

In [9]:
print(df.shape)
print("Missing wind:", df['wind_mw'].isna().mean())

(17520, 6)
Missing wind: 0.8090753424657534


In [10]:

df['wind_mw'] = df['wind_mw'].interpolate(method='linear')
df['wind_mw'] = df['wind_mw'].ffill()
df['wind_mw'] = df['wind_mw'].bfill()

print("Missing after fill:", df['wind_mw'].isna().mean())

Missing after fill: 0.0


In [11]:
print(df['wind_mw'].describe())

print(
    df.groupby('settlement_date')['settlement_period']
    .nunique()
    .value_counts()
)

count    17520.000000
mean      7310.086073
std       4382.018890
min        389.000000
25%       3478.716667
50%       6737.735294
75%      10744.036765
max      18620.000000
Name: wind_mw, dtype: float64
settlement_period
48    363
46      1
50      1
Name: count, dtype: int64


## 7. FILL WIND (SPARSE → CONTINUOUS)

In [12]:
# Sort chronologically (important for interpolation)
df = df.sort_values(['settlement_date', 'settlement_period']).reset_index(drop=True)

# Fill missing values
df['wind_mw'] = df['wind_mw'].interpolate(method='linear')
df['wind_mw'] = df['wind_mw'].ffill()
df['wind_mw'] = df['wind_mw'].bfill()

print("Missing wind (after fill):", df['wind_mw'].isna().mean())

Missing wind (after fill): 0.0


## 8. FINAL FEATURES

In [13]:
# Rename for modelling consistency (optional but cleaner)
df = df.rename(columns={
    'settlement_date': 'date',
    'settlement_period': 'sp'
})

## 9. FINAL VALIDATION

In [14]:
# Clean duplicate columns
df = df.loc[:, ~df.columns.duplicated()]

print("\nShape:", df.shape)

print("\nMissing values:")
print(df.isna().mean())

print("\nSettlement periods per day:")
sp_dist = df.groupby('date')['sp'].nunique()
print(sp_dist.value_counts())

print("\nWind summary:")
print(df['wind_mw'].describe())

# Assertions
assert len(df) == 17520, "❌ Row count incorrect"
assert df['wind_mw'].isna().mean().item() == 0, "❌ Wind still missing"
assert set(sp_dist.unique()).issubset({46, 48, 50}), "❌ Invalid SP structure"


Shape: (17520, 6)

Missing values:
date            0.0
sp              0.0
gspGroup        0.0
sw_demand_mw    0.0
sp_key          0.0
wind_mw         0.0
dtype: float64

Settlement periods per day:
sp
48    363
46      1
50      1
Name: count, dtype: int64

Wind summary:
count    17520.000000
mean      7310.086073
std       4382.018890
min        389.000000
25%       3478.716667
50%       6737.735294
75%      10744.036765
max      18620.000000
Name: wind_mw, dtype: float64


## 10. SAVE DATASET

In [15]:
output_path = DATA_PROCESSED + "modelling_dataset_2022.csv"

df.to_csv(output_path, index=False)

print(f"\nSaved → {output_path}")


Saved → ../data/processed/modelling_dataset_2022.csv


Final dataset contains:

- sp_key: settlement period identifier  
- sw_demand_mw: South West demand proxy  
- wind_mw: GB wind generation  
- date: settlement date  
- sp: settlement period  

This dataset forms the input to all subsequent modelling notebooks.

## 11.Summary

Wind generation is derived from intraday forecast vintages
(-2h to +6h lead time), collapsed to the latest available
forecast per settlement period.

Due to sparse forecast coverage (~80% missing), wind is
interpolated across settlement periods to produce a continuous
generation signal aligned to demand.